In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

CSV_PATH = Path("sales_clean.csv")
df = pd.read_csv(CSV_PATH, dtype={"customer_id": "string"})
print("Sisendi shape:", df.shape)
display(df.head())


Sisendi shape: (8684, 19)


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0
3,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva,2021-10-08,gold,1972.0
4,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu,2021-04-09,NaN,1996.0


In [2]:
# Viitekuupäev
today = pd.to_datetime('2025-02-28')


In [3]:
# Recency
df['sale_date'] = pd.to_datetime(df['sale_date'])
today = pd.to_datetime('2025-02-28')

recency = df.groupby('customer_id')['sale_date'].max().reset_index()
recency.columns = ['customer_id', 'last_purchase_date']
recency['Recency'] = (today - recency['last_purchase_date']).dt.days

In [4]:
# Frequency
frequency = df.groupby('customer_id')['sale_id'].count().reset_index()
frequency.columns = ['customer_id', 'Frequency']

In [8]:
# Monetary
monetary = df.groupby('customer_id')['total_price'].sum().reset_index()
monetary.columns = ['customer_id', 'Monetary']

In [9]:
rfm = pd.merge(recency[['customer_id', 'Recency']], frequency, on='customer_id')
rfm = pd.merge(rfm, monetary, on='customer_id')

# väiksem Recency = kõrgem skoor
rfm['R_score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# sama ostude arvuga kliendid
rfm['F_score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm['M_score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm.head()

,customer_id,Recency,Frequency,Monetary,R_score,F_score,M_score
0,2001.0,91,2,203.92,4,1,1
1,2004.0,71,2,1198.56,4,1,4
2,2005.0,148,4,959.60,3,4,4
3,2006.0,477,1,327.06,1,1,1
4,2007.0,29,1,318.63,5,1,1


In [10]:
# RFM skoor
rfm['RFM_Score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

rfm['Segment'] = pd.cut(
    rfm['RFM_Score'],
    bins=[2, 3, 6, 9, 12, 15],
    labels=['Lost', 'At Risk', 'Potential', 'Loyal', 'VIP Champions']
)

kokkuvote = rfm.groupby('Segment', observed=False).agg(
    Klientide_arv=('customer_id', 'count'),
    Keskmine_Recency=('Recency', 'mean'),
    Keskmine_Frequency=('Frequency', 'mean'),
    Keskmine_Monetary=('Monetary', 'mean'),
    Keskmine_RFM_Score=('RFM_Score', 'mean')
).round(2)

print(kokkuvote)

               Klientide_arv  Keskmine_Recency  Keskmine_Frequency  \
Segment                                                              
Lost                     124            537.40                1.00   
At Risk                  510            315.25                1.53   
Potential                766            211.38                2.42   
Loyal                    676            147.93                3.76   
VIP Champions            450             50.59                7.53   

               Keskmine_Monetary  Keskmine_RFM_Score  
Segment                                               
Lost                      160.73                3.00  
At Risk                   348.59                5.13  
Potential                 669.45                8.01  
Loyal                    1147.73               10.92  
VIP Champions            2453.83               13.85  


In [ ]:
# Salvestan RFM andmed
rfm.to_csv("rfm_results_clean.csv", index=False)